# 設定股票清單

In [2]:
from io import StringIO
import os
import pandas as pd
import numpy as np
import json
import requests
import time
from datetime import datetime

##################################################
# 設定爬取的日期範圍和股票清單
# 以年為單位, 會覆蓋掉舊檔

start_year = 2010
end_year = 2024

data = [
    # ["2330", "台積電", "tw"],
    # ["2454", "聯發科", "tw"],
    # ["2317", "鴻海", "tw" ],
    # ["2881", "富邦金", "tw" ],
    # ["2308", "台達電", "tw" ],
    # ["2882", "國泰金", "tw" ],
    # ["2412", "中華電", "tw" ],
    # ["2382", "廣達", "tw" ],
    ["2891", "中信金", "tw" ],
    ["3711", "日月光投控", "tw"], #10
    
    ["2886", "兆豐金", "tw" ],
    ["2303", "聯電", "tw" ],
    ["2357", "華碩", "tw" ],   
    ["2885", "元大金", "tw" ],  # 15
    ["2603", "長榮", "tw" ],     
    ["2884", "玉山金", "tw" ],
    ["1216", "統一", "tw" ],
    ["3045", "台灣大", "tw" ],
    ["2892", "第一金", "tw" ],
    ["2880", "華南金", "tw" ],   #20
    ["2002", "中鋼", "tw" ],
    ["2345", "智邦", "tw" ],
    ["5880", "合庫金", "tw" ],
    ["6505", "台塑化", "tw" ],
    ["6669", "緯穎", "tw" ],
    ["3008", "大立光", "tw" ],
    ["2395", "研華", "tw" ],
    ["2207", "和泰車", "tw" ],
    ["3034", "聯詠", "tw" ],
    ["3231", "緯創", "tw" ]      #30
]

##################################################

# 爬證交所資料

In [3]:
# 建立一個字典，其中包含所有欄位名稱的中文到英文的映射
column_name_mapping = {
    '日期': 'Date',
    '成交股數': 'Volume',
    '成交金額': 'Transaction Amount',
    '開盤價': 'Open',
    '最高價': 'High',
    '最低價': 'Low',
    '收盤價': 'Close',
    '漲跌價差': 'Diff',
    '成交筆數': 'Transaction Count'
}


def fetch_data_twse(ticker_symbol, year1, year2):
    print("Fetching data for " + ticker_symbol + " from " +
          str(year1) + " to " + str(year2) + "...")
    df_y = pd.DataFrame()
    for i in range(year1, year2+1):
        for j in range(12):
            month = str(i) + str(j+1).zfill(2)
            url = f'https://www.twse.com.tw/exchangeReport/STOCK_DAY?response=json&date={month}01&stockNo={ticker_symbol}'
            # print(url)
            data = requests.get(url).text
            data = json.loads(data)
            if data['stat'] != 'OK':
                print('Data not found for ' + month)
                break
            df_m = pd.DataFrame(data['data'])
            df_m.columns = data['fields']
            df_m = df_m.rename(columns=column_name_mapping)
            df_m.set_index('Date', inplace=True)
            df_y = pd.concat([df_y, df_m])

    # 將日期從民國年轉換為西元年
    df_y.index = pd.Series(df_y.index).str.replace(
        '/', '').apply(lambda x: str(int(x[:3]) + 1911) + x[3:])

    return df_y

# 畫技術分析


In [4]:
# 波動率
def historical_volatility(prices, window=10, trading_days=252):
    """
    計算歷史波動率 (Historical Volatility)

    :param prices: 一個包含價格數據的列表或Numpy數組
    :param window: 計算波動率的視窗期 (默認是252天)
    :param trading_days: 年化的交易天數，默認是252
    :return: 年化波動率
    """
    prices = np.asarray(prices)
    # 計算對數收益率
    log_returns = np.log(prices[1:] / prices[:-1])

    # 計算對數收益率的標準差
    std_dev = np.std(log_returns[-window:])

    # 年化波動率
    annualized_volatility = std_dev * np.sqrt(trading_days)
    
    return annualized_volatility

def cal_tech(id):

    path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')
    # folder_list = ['2330', '2454', '2317', '2382', '2308', '3711']

    stock_tech = pd.read_csv(os.path.join(path, id + '.csv'))

    # # 將stock_tech的值都換成float
    # stock_tech['Open'] = stock_tech['Open'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Close'] = stock_tech['Close'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['High'] = stock_tech['High'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Low'] = stock_tech['Low'].map(lambda x: float(x.replace(',', '')))

    # 計算移動均線 (SMA)
    stock_tech['MA5'] = stock_tech['Close'].rolling(window=5).mean()
    stock_tech['MA10'] = stock_tech['Close'].rolling(window=10).mean()
    stock_tech['MA20'] = stock_tech['Close'].rolling(window=20).mean()
    
    for i in range(1, 257):
        # 計算指數移動平均線 (EMA)
        stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()

    # # 計算指數移動平均線 (EMA)
    # stock_tech['EMA5'] = stock_tech['Close'].ewm(span=5, adjust=False).mean()
    # stock_tech['EMA12'] = stock_tech['Close'].ewm(span=12, adjust=False).mean()
    # stock_tech['EMA26'] = stock_tech['Close'].ewm(span=26, adjust=False).mean()
    # stock_tech['EMA60'] = stock_tech['Close'].ewm(span=60, adjust=False).mean()

    # # 計算MACD和DIF
    # stock_tech['DIF'] = stock_tech['EMA12'] - stock_tech['EMA26']
    # stock_tech['MACD'] = stock_tech['DIF'].ewm(span=9, adjust=False).mean()

    # # 計算RSI

    # def calculate_rsi(series, period=14):
    #     delta = series.diff(1)
    #     gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    #     loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    #     rs = gain / loss
    #     rsi = 100 - (100 / (1 + rs))
    #     return rsi

    # stock_tech['RSI'] = calculate_rsi(stock_tech['Close'])

    # # 計算KDJ
    # low_list = stock_tech['Low'].rolling(9, min_periods=9).min()
    # low_list.fillna(value=stock_tech['Low'].expanding().min(), inplace=True)
    # high_list = stock_tech['High'].rolling(9, min_periods=9).max()
    # high_list.fillna(value=stock_tech['High'].expanding().max(), inplace=True)
    # rsv = (stock_tech['Close'] - low_list) / (high_list - low_list) * 100

    # # 初始值設定
    # stock_tech['K'] = 50
    # stock_tech['D'] = 50
    

    # for i in range(1, len(stock_tech)):
    #     stock_tech.loc[stock_tech.index[i], 'K'] = (
    #         2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
    #     stock_tech.loc[stock_tech.index[i], 'D'] = (
    #         2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']
        
    # stock_tech['Volatility'] = stock_tech['Close'].rolling(window=252).apply(historical_volatility, raw=True)

    # 顯示DataFrame
    stock_tech.to_csv(os.path.join(path, id + 'tech.csv'),
                      encoding='utf-8', index=False)

# 主迴圈

In [5]:
path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')
for stock_data in data:
    id = stock_data[0]
    
    try:
        
        # 抓取資料
        stock_tech = fetch_data_twse(id, start_year, end_year)  # 這裡可以自行設定要抓取的時間範圍
        # Remove commas and convert to numeric types
        df = stock_tech.map(lambda x: pd.to_numeric(
            str(x).replace(",", "").replace('"', ''), errors='coerce'))
        df.to_csv(os.path.join(path, id + '.csv'))  # 將資料存成 csv 檔
        print('save ' + id + '.csv')
        
        # 載入資料，並畫技術指標
        cal_tech(id)
        print('save ' + id + 'tech.csv')
    except Exception as e:
        print("Error: " + id + " " + str(e))
        pass

Fetching data for 2891 from 2010 to 2024...
save 2891.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame

save 2891tech.csv
Fetching data for 3711 from 2010 to 2024...
Data not found for 201001
Data not found for 201101
Data not found for 201201
Data not found for 201301
Data not found for 201401
Data not found for 201501
Data not found for 201601
Data not found for 201701
Data not found for 201801
save 3711.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame

save 3711tech.csv
Fetching data for 2886 from 2010 to 2024...
save 2886.csv


...

save 2207tech.csv
Fetching data for 3034 from 2010 to 2024...
save 3034.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame

save 3034tech.csv
Fetching data for 3231 from 2010 to 2024...
save 3231.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stock_tech['EMA' + str(i)] = stock_tech['Close'].ewm(span=i, adjust=False).mean()
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_23929/3550219514.py:43: PerformanceWarning: DataFrame

save 3231tech.csv
